# pufbench: run one session of the benchmark

Runs part of the paper's design (Sections IV-A and IV-C) and saves every result
to Google Drive as soon as it finishes. **If Colab disconnects, run all cells
again**: finished runs are skipped and the session continues where it stopped.

1. Run cells 1-3 once per session.
2. In cell 4, set `SESSION` to the session you want to run.
3. Run cells 4-6.

## 1. Connect Google Drive (results are saved here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RESULTS_DIR = '/content/drive/MyDrive/pufbench_results'
import os
os.makedirs(RESULTS_DIR, exist_ok=True)
print('results folder:', RESULTS_DIR)

## 2. Get the code from GitHub (or update it)

In [ ]:
REPO = 'https://github.com/kinayatphdee/pufbench.git'
# Private repository instead? Create a token on GitHub (Settings > Developer settings >
# Personal access tokens), then use:
# REPO = 'https://<YOUR_TOKEN>@github.com/kinayatphdee/pufbench.git'

import os
if os.path.exists('/content/pufbench'):
    !cd /content/pufbench && git pull -q
else:
    !git clone -q {REPO} /content/pufbench
%cd /content/pufbench
!git log -1 --format='code version: %h  %s'

## 3. Quick check that the code works here (about 1 minute)

In [ ]:
!pip install -q pytest
!python -m pytest -q tests/test_sim.py tests/test_lr.py tests/test_runner_design.py

## 4. Choose the session

| Session | What | Rough time |
|---|---|---|
| A1 | LR, all families, seed 0 | ~2 h |
| A2 | CMA-ES, arbiter | ~2.5 h |
| A3 | CMA-ES, 3-XOR | ~8 h |
| A4 | CMA-ES, feed-forward | ~6 h |
| A5 | MLP, all families, seed 0 | ~5 h |
| B1 | LR, all families, seeds 1-4 | ~7 h |
| B2 | MLP, arbiter, seeds 1-4 | ~6 h |
| B3 | MLP, 3-XOR and 4-XOR, seeds 1-4 | ~7 h |
| B4 | MLP, 5-XOR and feed-forward, seeds 1-4 | ~7 h |

Phase A (A1-A5) gives a complete first version of every result. Times are rough upper bounds.

In [ ]:
SESSION = 'A1'      # <- change this

SESSIONS = {
    'A1': '--attackers lr  --seeds 0',
    'A2': '--attackers cma --families arbiter',
    'A3': '--attackers cma --families xor3',
    'A4': '--attackers cma --families ff',
    'A5': '--attackers mlp --seeds 0',
    'B1': '--attackers lr  --seeds 1 2 3 4',
    'B2': '--attackers mlp --seeds 1 2 3 4 --families arbiter',
    'B3': '--attackers mlp --seeds 1 2 3 4 --families xor3 xor4',
    'B4': '--attackers mlp --seeds 1 2 3 4 --families xor5 ff',
}
ARGS = SESSIONS[SESSION]
OUT = f'{RESULTS_DIR}/{SESSION}.csv'
print('session', SESSION, '|', ARGS, '| results ->', OUT)

## 5. How many runs are planned, and how many are left

In [ ]:
!python -m pufbench.runner --design paper {ARGS} --out "{OUT}" --count

## 6. Run the session

Prints one line per finished run. Every line is already saved in Drive. If Colab
disconnects, reconnect and run all cells again (with the same `SESSION`).

In [ ]:
!python -u -m pufbench.runner --design paper {ARGS} --out "{OUT}"